In [18]:
# --------------------------------------------
# STEP 1: Install Required Libraries
# --------------------------------------------

!pip install librosa kagglehub tensorflow --quiet

In [19]:
# =========================================================
# STEP 2: IMPORT LIBRARIES
# =========================================================

import os
import numpy as np
import librosa
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

In [25]:
# =========================================================
# STEP 3: DOWNLOAD DATASET
# =========================================================

path = kagglehub.dataset_download(
    "kongaevans/speaker-recognition-dataset"
)

print("Dataset Downloaded At:")
print(path)

Using Colab cache for faster access to the 'speaker-recognition-dataset' dataset.
Dataset Downloaded At:
/kaggle/input/speaker-recognition-dataset


In [48]:
# =========================================================
# STEP 4: SET CORRECT DATASET PATH
# =========================================================

dataset_path = os.path.join(
    path,
    "16000_pcm_speeches"
)

print("Dataset Path:")
print(dataset_path)

Dataset Path:
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches


In [49]:
# =========================================================
# STEP 5: SELECT ONLY 3 CLASSES
# =========================================================
speaker_1 = "Jens_Stoltenberg"

speaker_2 = "Benjamin_Netanyau"

unknown_speakers = [
    "Nelson_Mandela",
    "Julia_Gillard",
    "Magaret_Tarcher"
]
# =========================================================
# STEP 6: FEATURE EXTRACTION SETTINGS
# =========================================================

N_MFCC = 40
MAX_PAD_LEN = 100

In [50]:
# =========================================================
# STEP 7: MFCC FEATURE EXTRACTION FUNCTION
# =========================================================

def extract_features(file_path):

    try:

        # Load audio
        audio, sample_rate = librosa.load(
            file_path,
            sr=16000
        )

        # Extract MFCC
        mfccs = librosa.feature.mfcc(
            y=audio,
            sr=sample_rate,
            n_mfcc=N_MFCC
        )

        # Padding / Trimming
        if mfccs.shape[1] < MAX_PAD_LEN:

            pad_width = MAX_PAD_LEN - mfccs.shape[1]

            mfccs = np.pad(
                mfccs,
                pad_width=((0, 0), (0, pad_width)),
                mode='constant'
            )

        else:

            mfccs = mfccs[:, :MAX_PAD_LEN]

        return mfccs.flatten()

    except Exception as e:

        print("Error Processing:", file_path)
        print(e)

        return None


In [51]:
# =========================================================
# STEP 8: LOAD DATASET
# =========================================================

X = []
y = []

# -----------------------------
# PERSON 1
# -----------------------------

speaker_folder = os.path.join(
    dataset_path,
    speaker_1
)

for file_name in os.listdir(speaker_folder):

    if file_name.lower().endswith(".wav"):

        file_path = os.path.join(
            speaker_folder,
            file_name
        )

        features = extract_features(file_path)

        if features is not None:

            X.append(features)
            y.append("person1")

# -----------------------------
# PERSON 2
# -----------------------------

speaker_folder = os.path.join(
    dataset_path,
    speaker_2
)

for file_name in os.listdir(speaker_folder):

    if file_name.lower().endswith(".wav"):

        file_path = os.path.join(
            speaker_folder,
            file_name
        )

        features = extract_features(file_path)

        if features is not None:

            X.append(features)
            y.append("person2")

# -----------------------------
# UNKNOWN SPEAKERS
# -----------------------------

for unknown_speaker in unknown_speakers:

    speaker_folder = os.path.join(
        dataset_path,
        unknown_speaker
    )

    for file_name in os.listdir(speaker_folder):

        if file_name.lower().endswith(".wav"):

            file_path = os.path.join(
                speaker_folder,
                file_name
            )

            features = extract_features(file_path)

            if features is not None:

                X.append(features)
                y.append("unknown")

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)


In [52]:

print("\n===================================")
print("DATA LOADED SUCCESSFULLY")
print("===================================")

print("X Shape:", X.shape)
print("y Shape:", y.shape)

from collections import Counter

print("\nClass Distribution:")
print(Counter(y))


DATA LOADED SUCCESSFULLY
X Shape: (7501, 4000)
y Shape: (7501,)

Class Distribution:
Counter({np.str_('unknown'): 4501, np.str_('person1'): 1500, np.str_('person2'): 1500})


In [53]:
# =========================================================
# STEP 10: ENCODE LABELS
# =========================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# One-hot encoding
y_categorical = to_categorical(y_encoded)

print("\nEncoded Classes:")
print(label_encoder.classes_)


Encoded Classes:
['person1' 'person2' 'unknown']


In [54]:
# =========================================================
# STEP 11: TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("\nTraining Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)



Training Shape: (6000, 4000)
Testing Shape: (1501, 4000)


In [55]:
# =========================================================
# STEP 12: BUILD DEEP LEARNING MODEL
# =========================================================

model = Sequential([

    Dense(
        256,
        activation='relu',
        input_shape=(X.shape[1],)
    ),

    Dropout(0.3),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dense(
        3,
        activation='softmax'
    )
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [56]:
# =========================================================
# STEP 13: COMPILE MODEL
# =========================================================

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =========================================================
# STEP 14: MODEL SUMMARY
# =========================================================

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 256)            │     1,024,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,065,603 (4.06 MB)

 Trainable params: 1,065,603 (4.06 MB)

 Non-trainable params: 0 (0.00 B)

In [57]:
# =========================================================
# STEP 15: TRAIN MODEL
# =========================================================

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test)
)

# =========================================================
# STEP 16: EVALUATE MODEL
# =========================================================

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\n===================================")
print("TEST ACCURACY:", accuracy)
print("===================================")

# =========================================================
# STEP 17: SAVE MODEL
# =========================================================

model.save("speaker_recognition_model.h5")

print("\nModel Saved Successfully!")


Epoch 1/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.8318 - loss: 1.4076 - val_accuracy: 0.9647 - val_loss: 0.1082
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9502 - loss: 0.1904 - val_accuracy: 0.9887 - val_loss: 0.0427
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.9647 - loss: 0.1284 - val_accuracy: 0.9900 - val_loss: 0.0386
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9732 - loss: 0.0808 - val_accuracy: 0.9933 - val_loss: 0.0337
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9702 - loss: 0.1062 - val_accuracy: 0.9873 - val_loss: 0.0312
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.9782 - loss: 0.0662 - val_accuracy: 0.9940 - val_loss: 0.0243
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9807 - loss: 0.0655 - val_accuracy: 0.9927 - val_loss: 0.0247
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9807 - loss: 0.0602 - val_accu


TEST ACCURACY: 0.9940040111541748

Model Saved Successfully!


In [58]:
model.save("speaker_recognition_model.keras")

In [59]:
# =========================================================
# STEP 18: PREDICTION FUNCTION
# =========================================================

def predict_speaker(file_path):

    features = extract_features(file_path)

    if features is None:
        return "Error Processing Audio"

    features = np.expand_dims(features, axis=0)

    prediction = model.predict(features)

    predicted_index = np.argmax(prediction)

    confidence = np.max(prediction)

    predicted_label = label_encoder.inverse_transform(
        [predicted_index]
    )[0]

    print("\n===================================")
    print("PREDICTION RESULT")
    print("===================================")

    print("Predicted Speaker :", predicted_label)
    print("Confidence Score  :", confidence)


predict_speaker("/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Julia_Gillard/662.wav")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step

PREDICTION RESULT
Predicted Speaker : unknown
Confidence Score  : 1.0


code to get the persons audio.

In [41]:
# =========================================================
# SHOW 5 AUDIO FILES FROM EACH CLASS
# =========================================================

import os
import random

# Dataset path
dataset_path = os.path.join(
    path,
    "16000_pcm_speeches"
)

# Selected speakers
selected_folders = {
    "person1": "Jens_Stoltenberg",
    "person2": "Benjamin_Netanyau",
    "unknown_1": "Nelson_Mandela",
    "unknown_2": "Julia_Gillard",
    "unknown_3": "Magaret_Tarcher"
}

# =========================================================
# DISPLAY 5 FILES FROM EACH FOLDER
# =========================================================

for label, folder_name in selected_folders.items():

    folder_path = os.path.join(
        dataset_path,
        folder_name
    )

    print("\n===================================")
    print(f"CLASS: {label}")
    print(f"FOLDER: {folder_name}")
    print("===================================")

    # Get all wav files
    wav_files = [
        file for file in os.listdir(folder_path)
        if file.lower().endswith(".wav")
    ]

    # Randomly select 5 files
    sample_files = random.sample(
        wav_files,
        min(5, len(wav_files))
    )

    # Print file names
    for file_name in sample_files:

        full_path = os.path.join(
            folder_path,
            file_name
        )

        print(full_path)


CLASS: person1
FOLDER: Jens_Stoltenberg
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Jens_Stoltenberg/1353.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Jens_Stoltenberg/1049.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Jens_Stoltenberg/1071.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Jens_Stoltenberg/1361.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Jens_Stoltenberg/613.wav

CLASS: person2
FOLDER: Benjamin_Netanyau
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Benjamin_Netanyau/984.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Benjamin_Netanyau/1459.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Benjamin_Netanyau/82.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Benjamin_Netanyau/0.wav
/kaggle/input/speaker-recognition-dataset/16000_pcm_speeches/Benjamin_Netanyau/94.wav

CLASS: unknown_1
FOLDER: Nelson_Mandela
/kaggle/in